In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars seaborn

import polars as pl

In [3]:
!nvidia-smi

Wed Apr 22 15:06:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D6:00.0 Off |                  Off |
| 30%   53C    P2            132W /  230W |    4903MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [6]:
df['cc_finding_categories'].value_counts().sort_index()

cc_finding_categories
['Architectural Distortion', 'Mass']                                                                   1
['Architectural Distortion']                                                                          40
['Asymmetry', 'Mass']                                                                                  1
['Asymmetry']                                                                                         20
['Focal Asymmetry']                                                                                  107
['Global Asymmetry']                                                                                  11
['Mass']                                                                                             443
['Nipple Retraction', 'Mass']                                                                          1
['Nipple Retraction', 'Skin Thickening', 'Mass']                                                       1
['Nipple Retraction']            

In [7]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [8]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [9]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['label'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,5,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [10]:
inbreast_df['label'].value_counts()

label
1    98
0    30
4    28
3    20
2    11
Name: count, dtype: int64

In [11]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['label'] = df['cc_breast_birads'].apply(birads_to_label)

In [12]:
df['label'].value_counts()

label
0    6703
1    2337
3     339
2     319
4     112
Name: count, dtype: int64

In [13]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [14]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [15]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [16]:
train_df.shape

(7057, 32)

In [17]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.9, 1.1),
                contrast=(0.9, 1.1)
            )
        ], p=0.6),
        
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.03)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [18]:
import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=None,
                scale=(0.9, 1.1),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))
        ], p=0.2),
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.01, 0.04), ratio=(0.3, 3.0), value=0),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [19]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "birads_0", 1: "birads_1",2: "birads_2",3: "birads_3",4: "birads_4"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]


    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        qry_label   = row['label']
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
        # qry_device  = row['device_id']

        qry_im = Image.open(row['merged_image_path']).convert("RGB")

        if self.transform:
            qry_im = self.transform(qry_im)

        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )

        return {
            "qry_im":      qry_im,
            "qry_gt":      torch.tensor(qry_label,   dtype=torch.long),
            "finding":     qry_finding,
            "finding_vec": finding_vec,
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long)
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)

    labels                       = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, 
        # shuffle=True,
        sampler=sampler,
        drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [20]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

Class counts: {np.int64(0): np.int64(4824), np.int64(1): np.int64(1674), np.int64(2): np.int64(229), np.int64(3): np.int64(247), np.int64(4): np.int64(83)}
Smoothed class weights: [0.259 0.439 1.187 1.143 1.972]


In [21]:
# finding columns distribution
finding_cols = ['has_neg_b1','has_neg_b2','has_mass','has_calc','has_structural','has_lymph']
print(train_df[finding_cols].sum())

has_neg_b1        4824
has_neg_b2        1666
has_mass           414
has_calc           146
has_structural      72
has_lymph           18
dtype: int64


In [22]:
# THE KEY — joint distribution: for each finding, how many samples per BI-RADS class
for f in finding_cols:
    print(f"\n--- {f} ---")
    print(train_df[train_df[f]==1]['label'].value_counts().sort_index())


--- has_neg_b1 ---
label
0    4824
Name: count, dtype: int64

--- has_neg_b2 ---
label
1    1666
Name: count, dtype: int64

--- has_mass ---
label
2    195
3    156
4     63
Name: count, dtype: int64

--- has_calc ---
label
2    24
3    78
4    44
Name: count, dtype: int64

--- has_structural ---
label
1     7
2    12
3    39
4    14
Name: count, dtype: int64

--- has_lymph ---
label
1    1
3    9
4    8
Name: count, dtype: int64


In [ ]:
import os
import gc
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# Config
# =============================================================================

FINDING_COLS = [
    "has_neg_b1",       # finding 0
    "has_neg_b2",       # finding 1
    "has_mass",         # finding 2
    "has_calc",         # finding 3
    "has_structural",   # finding 4
    "has_lymph",        # finding 5
]

NUM_FINDINGS       = 6
NUM_CLASSES        = 5
BIRADS_CLASS_NAMES = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]

ALLOWED_BIRADS_BY_FINDING = torch.tensor([
    [1, 0, 0, 0, 0],   # finding 0
    [0, 1, 0, 0, 0],   # finding 1
    [0, 0, 1, 1, 1],   # mass
    [0, 0, 1, 1, 1],   # calc
    [0, 0, 1, 1, 1],   # structural
    [0, 0, 1, 1, 1],   # lymph
], dtype=torch.float32)

FINDING_PROTO_MOMENTUM = [
    0.999,  # finding 0
    0.997,  # finding 1
    0.980,  # finding 2
    0.960,  # finding 3
    0.920,  # finding 4
    0.850,  # finding 5
]

BIRADS_PROTO_MOMENTUM = [
    0.999,  # BI-RADS 1
    0.997,  # BI-RADS 2
    0.970,  # BI-RADS 3
    0.960,  # BI-RADS 4
    0.920,  # BI-RADS 5
]

# -----------------------------------------------------------------------------
# Sample weights
# -----------------------------------------------------------------------------
FINDING_SAMPLE_WEIGHT = [
    0.1,   # finding 0
    0.15,   # finding 1
    1.0,   # finding 2
    1.0,   # finding 3
    1.0,   # finding 4
    1.0,   # finding 5
]

BIRADS_SAMPLE_WEIGHT = [
    0.1,   # BI-RADS 1
    0.15,   # BI-RADS 2
    1.0,   # BI-RADS 3
    1.0,   # BI-RADS 4
    1.0,   # BI-RADS 5
]

PROTO_MIN_UPDATES = 15


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# =============================================================================
# Helpers
# =============================================================================

def labels_to_levels(labels, num_classes):
    B = labels.size(0)
    levels = torch.zeros(B, num_classes - 1, device=labels.device, dtype=torch.float32)
    for k in range(num_classes - 1):
        levels[:, k] = (labels > k).float()
    return levels


def ordinal_logits_to_label(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)


def ordinal_logits_to_probs(logits):
    q = torch.sigmoid(logits)
    B = q.size(0)
    K = q.size(1) + 1
    probs = torch.zeros(B, K, device=logits.device, dtype=logits.dtype)

    probs[:, 0] = 1.0 - q[:, 0]
    for c in range(1, K - 1):
        probs[:, c] = q[:, c - 1] - q[:, c]
    probs[:, K - 1] = q[:, K - 2]
    return probs.clamp(min=1e-8, max=1.0)


# =============================================================================
# Pooling
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


# =============================================================================
# Lightweight Swin-style block on P5
# =============================================================================

class MLPBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, dim)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.drop2(x)
        return x


class WindowAttention(nn.Module):
    """
    Lightweight window attention.
    Input/Output: [B, C, H, W]
    """
    def __init__(self, dim, num_heads=4, window_size=7, dropout=0.0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLPBlock(dim, mlp_ratio=4.0, dropout=dropout)

    def _window_partition(self, x):
        B, C, H, W = x.shape
        ws = self.window_size

        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h))

        Hp, Wp = x.shape[-2], x.shape[-1]
        x = x.view(B, C, Hp // ws, ws, Wp // ws, ws)
        x = x.permute(0, 2, 4, 3, 5, 1).contiguous()
        windows = x.view(-1, ws * ws, C)
        return windows, Hp, Wp, pad_h, pad_w

    def _window_reverse(self, windows, Hp, Wp, B):
        ws = self.window_size
        C = windows.shape[-1]
        nH = Hp // ws
        nW = Wp // ws

        x = windows.view(B, nH, nW, ws, ws, C)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous()
        x = x.view(B, C, Hp, Wp)
        return x

    def forward(self, x):
        B, C, H, W = x.shape
        windows, Hp, Wp, pad_h, pad_w = self._window_partition(x)

        shortcut = windows
        w = self.norm1(windows)
        attn_out, _ = self.attn(w, w, w, need_weights=False)
        windows = shortcut + attn_out
        windows = windows + self.mlp(self.norm2(windows))

        x = self._window_reverse(windows, Hp, Wp, B)

        if pad_h > 0 or pad_w > 0:
            x = x[:, :, :H, :W]

        return x


# =============================================================================
# Cross-scale attention
# =============================================================================

class CrossScaleAttention(nn.Module):
    """
    Query comes from target feature map.
    Key/Value comes from source feature map.
    Output shape = target shape.
    """
    def __init__(self, dim, num_heads=4, dropout=0.0):
        super().__init__()
        self.norm_q  = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.out_norm = nn.LayerNorm(dim)
        self.mlp = MLPBlock(dim, mlp_ratio=4.0, dropout=dropout)

    def forward(self, target_feat, source_feat):
        B, C, Ht, Wt = target_feat.shape
        _, _, Hs, Ws = source_feat.shape

        q = target_feat.flatten(2).transpose(1, 2)   # [B, Ht*Wt, C]
        kv = source_feat.flatten(2).transpose(1, 2)  # [B, Hs*Ws, C]

        qn = self.norm_q(q)
        kvn = self.norm_kv(kv)

        attn_out, _ = self.attn(qn, kvn, kvn, need_weights=False)
        x = q + attn_out
        x = x + self.mlp(self.out_norm(x))

        x = x.transpose(1, 2).reshape(B, C, Ht, Wt)
        return x


# =============================================================================
# Global context block
# =============================================================================

class GlobalContextBlock(nn.Module):
    """
    Produces one global token and projects it back.
    """
    def __init__(self, dim, reduction=4):
        super().__init__()
        hidden = max(dim // reduction, 64)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, dim),
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, C, _, _ = x.shape
        g = self.pool(x).view(B, C)
        g = self.mlp(self.norm(g))
        return g


# =============================================================================
# Backbone feature extractors
# =============================================================================

class EfficientNetB3FeatureExtractor(nn.Module):
    """
    EfficientNet-B3:
      p5 = last feature stage
      p4 = second last feature stage
    """
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features

    def forward(self, x):
        feats = []
        for block in self.features:
            x = block(x)
            feats.append(x)

        p5 = feats[-1]
        p4 = feats[-2]
        return p4, p5


class ConvNeXtBaseFeatureExtractor(nn.Module):
    """
    ConvNeXt-Base:
      p5 = last stage
      p4 = last-3 stage
    Because last-2 is the downsample op, not the semantic stage.
    """
    def __init__(self, backbone):
        super().__init__()
        self.features = backbone.features

    def forward(self, x):
        feats = []
        for block in self.features:
            x = block(x)
            feats.append(x)

        p5 = feats[-1]
        p4 = feats[-3]
        return p4, p5


# =============================================================================
# Hybrid encoder: P4 + P5 + Swin on P5 + Cross-scale attention + Global context
# =============================================================================

class HybridMassAwareEncoder(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
        num_classes=5,
        fuse_dim=256,
        swin_heads=4,
        swin_window=7,
        swin_depth=2,
        cross_heads=4,
        dummy_size=224,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.num_classes = num_classes
        self.fuse_dim = fuse_dim

        if "efficientnet_b3" in self.backbone_name:
            self.feature_extractor = EfficientNetB3FeatureExtractor(backbone)
        elif "convnext_base" in self.backbone_name:
            self.feature_extractor = ConvNeXtBaseFeatureExtractor(backbone)
        else:
            raise ValueError(f"Unsupported hybrid backbone: {backbone_name}")

        # infer channels
        with torch.no_grad():
            was_training = self.feature_extractor.training
            self.feature_extractor.eval()
            dummy = torch.zeros(1, 3, dummy_size, dummy_size)
            p4, p5 = self.feature_extractor(dummy)
            p4_in = p4.shape[1]
            p5_in = p5.shape[1]
            self.feature_extractor.train(was_training)

        self.p4_proj = nn.Sequential(
            nn.Conv2d(p4_in, fuse_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(fuse_dim),
            nn.GELU(),
        )

        self.p5_proj = nn.Sequential(
            nn.Conv2d(p5_in, fuse_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(fuse_dim),
            nn.GELU(),
        )

        # local semantic refinement on P5
        self.p5_swin = nn.Sequential(*[
            WindowAttention(
                dim=fuse_dim,
                num_heads=swin_heads,
                window_size=swin_window,
                dropout=0.0,
            ) for _ in range(swin_depth)
        ])

        # bidirectional cross-scale attention
        self.p4_from_p5 = CrossScaleAttention(
            dim=fuse_dim,
            num_heads=cross_heads,
            dropout=0.0,
        )
        self.p5_from_p4 = CrossScaleAttention(
            dim=fuse_dim,
            num_heads=cross_heads,
            dropout=0.0,
        )

        # global context
        self.global_context = GlobalContextBlock(fuse_dim)

        # fusion
        self.fuse = nn.Sequential(
            nn.Conv2d(fuse_dim * 3, fuse_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(fuse_dim),
            nn.GELU(),
            nn.Conv2d(fuse_dim, fuse_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(fuse_dim),
            nn.GELU(),
            nn.Conv2d(fuse_dim, fuse_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(fuse_dim),
            nn.GELU(),
        )

        self.pool = AttentionPool2d(fuse_dim)

        # combine pooled spatial vector + global token
        self.final_proj = nn.Sequential(
            nn.Linear(fuse_dim * 2, fuse_dim),
            nn.LayerNorm(fuse_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.cls_head = nn.Sequential(
            nn.Linear(fuse_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes - 1),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(fuse_dim, fuse_dim),
            nn.LayerNorm(fuse_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fuse_dim, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(fuse_dim, fuse_dim),
            nn.LayerNorm(fuse_dim),
            nn.ReLU(inplace=True),
            nn.Linear(fuse_dim, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        modules = [
            self.p4_proj, self.p5_proj, self.fuse,
            self.final_proj, self.cls_head,
            self.proj_head_finding, self.proj_head_birads
        ]
        for mod in modules:
            for m in mod.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                    if getattr(m, "bias", None) is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm, nn.BatchNorm1d)):
                    if hasattr(m, "weight") and m.weight is not None:
                        nn.init.ones_(m.weight)
                    if hasattr(m, "bias") and m.bias is not None:
                        nn.init.zeros_(m.bias)

    def forward_features(self, x, return_maps=False):
        p4, p5 = self.feature_extractor(x)

        # project
        p4 = self.p4_proj(p4)
        p5 = self.p5_proj(p5)

        # local attention on semantic map
        p5 = self.p5_swin(p5)

        # bidirectional cross-scale attention
        p4_att = self.p4_from_p5(p4, p5)   # p4 queries p5
        p5_att = self.p5_from_p4(p5, p4)   # p5 queries p4

        # align p5 to p4 size
        p5_up = F.interpolate(
            p5_att, size=p4_att.shape[-2:],
            mode="bilinear", align_corners=False
        )

        # global token from combined semantics
        global_seed = 0.5 * (p4_att + p5_up)
        g = self.global_context(global_seed)                     # [B, C]
        g_map = g.unsqueeze(-1).unsqueeze(-1).expand_as(p4_att) # [B, C, H, W]

        # fuse p4 + p5 + global
        fused = self.fuse(torch.cat([p4_att, p5_up, g_map], dim=1))

        # spatial attentive pooling + global token
        pooled = self.pool(fused)                               # [B, C]
        feat = self.final_proj(torch.cat([pooled, g], dim=1))   # [B, C]

        if return_maps:
            return feat, p4_att, p5_att, fused
        return feat

    def forward(self, x, return_embeddings=False, return_maps=False):
        if return_maps:
            feat, p4, p5, fused = self.forward_features(x, return_maps=True)
        else:
            feat = self.forward_features(x, return_maps=False)

        logits = self.cls_head(feat)

        if return_embeddings and return_maps:
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)
            emb_b = F.normalize(self.proj_head_birads(feat), dim=1)
            return logits, emb_f, emb_b, p4, p5, fused

        if return_embeddings:
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)
            emb_b = F.normalize(self.proj_head_birads(feat), dim=1)
            return logits, emb_f, emb_b

        return logits


# =============================================================================
# Dual prototype wrapper
# =============================================================================

class DualProtoHybridNet(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.4,
        num_classes=5,
        num_findings=NUM_FINDINGS,
        temperature=0.07,
        fuse_dim=256,
        swin_heads=4,
        swin_window=7,
        swin_depth=2,
        cross_heads=4,
    ):
        super().__init__()
        self.num_classes  = num_classes
        self.num_findings = num_findings
        self.temperature  = temperature
        self.emb_dim      = emb_dim

        backbone = backbone_fn(weights=backbone_weights)

        self.encoder = HybridMassAwareEncoder(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
            fuse_dim=fuse_dim,
            swin_heads=swin_heads,
            swin_window=swin_window,
            swin_depth=swin_depth,
            cross_heads=cross_heads,
        )

        self.register_buffer(
            "proto_finding",
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_finding_updates",
            torch.zeros(num_findings, dtype=torch.long),
        )

        self.register_buffer(
            "proto_birads",
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_birads_updates",
            torch.zeros(num_classes, dtype=torch.long),
        )

        self.register_buffer(
            "finding_momentum",
            torch.tensor(FINDING_PROTO_MOMENTUM, dtype=torch.float),
        )
        self.register_buffer(
            "birads_momentum",
            torch.tensor(BIRADS_PROTO_MOMENTUM, dtype=torch.float),
        )

        self.register_buffer(
            "allowed_birads_by_finding",
            ALLOWED_BIRADS_BY_FINDING.clone(),
        )

    @torch.no_grad()
    def update_finding_prototypes(self, emb_f, finding_vec):
        for f in range(self.num_findings):
            mask = finding_vec[:, f] > 0.5
            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_f[mask].mean(dim=0), dim=0)
            m = self.finding_momentum[f].item()

            if self.proto_finding_updates[f] == 0:
                self.proto_finding[f] = batch_proto
            else:
                self.proto_finding[f] = F.normalize(
                    m * self.proto_finding[f] + (1.0 - m) * batch_proto,
                    dim=0,
                )
            self.proto_finding_updates[f] += 1

    @torch.no_grad()
    def update_birads_prototypes(self, emb_b, labels):
        for c in range(self.num_classes):
            mask = labels == c
            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_b[mask].mean(dim=0), dim=0)
            m = self.birads_momentum[c].item()

            if self.proto_birads_updates[c] == 0:
                self.proto_birads[c] = batch_proto
            else:
                self.proto_birads[c] = F.normalize(
                    m * self.proto_birads[c] + (1.0 - m) * batch_proto,
                    dim=0,
                )
            self.proto_birads_updates[c] += 1

    def build_allowed_birads_mask(self, finding_vec):
        allowed = finding_vec @ self.allowed_birads_by_finding.to(finding_vec.device)
        return (allowed > 0).float()

    def forward_encoder(self, x, return_embeddings=False):
        return self.encoder(x, return_embeddings=return_embeddings)


# =============================================================================
# Losses
# =============================================================================

class MultiPositiveProtoInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, pos_mask, sample_weights):
        ready = (proto_updates >= self.min_updates)
        if ready.sum() < 2:
            return torch.tensor(0.0, device=q.device, requires_grad=False)

        losses = []
        weights = []

        for i in range(q.size(0)):
            pos_i = (pos_mask[i] > 0.5) & ready
            if pos_i.sum() == 0:
                continue

            sims = torch.matmul(prototypes[ready], q[i]) / self.tau
            pos_ready_mask = pos_i[ready]

            if pos_ready_mask.sum() == 0:
                continue
            if pos_ready_mask.sum() == ready.sum():
                continue

            log_den = torch.logsumexp(sims, dim=0)
            log_num = torch.logsumexp(sims[pos_ready_mask], dim=0)
            loss_i = -(log_num - log_den)

            losses.append(loss_i)
            weights.append(sample_weights[i])

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, requires_grad=False)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)
        return (losses * weights).sum() / (weights.sum() + 1e-8)


class WeightedCORALLoss(nn.Module):
    def __init__(
        self,
        num_classes=5,
        class_weights=None,
        underestimation_weight=2.5,
        level_weights=None,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.underestimation_weight = underestimation_weight

        if class_weights is None:
            class_weights = [0.1, 0.15, 1.50, 1.50, 1.80]
        self.register_buffer(
            "class_weights",
            torch.tensor(class_weights, dtype=torch.float32)
        )

        if level_weights is None:
            level_weights = [0.5, 1.2, 1.5, 1.8]
        self.register_buffer(
            "level_weights",
            torch.tensor(level_weights, dtype=torch.float32)
        )

    def forward(self, logits, labels):
        labels = labels.long()
        levels = labels_to_levels(labels, self.num_classes)

        bce = F.binary_cross_entropy_with_logits(
            logits, levels, reduction="none"
        )

        bce = bce * self.level_weights.view(1, -1)

        probs = torch.sigmoid(logits)
        under_mask = (levels == 1.0) & (probs < 0.5)
        under_scale = torch.ones_like(bce)
        under_scale[under_mask] = self.underestimation_weight
        bce = bce * under_scale

        per_sample = bce.mean(dim=1)
        sample_w = self.class_weights[labels]
        return (per_sample * sample_w).sum() / (sample_w.sum() + 1e-8)


class HierarchicalBiradsConsistencyLoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, emb_b, proto_birads, proto_updates, allowed_mask, sample_weights):
        ready = (proto_updates >= self.min_updates)

        losses = []
        weights = []

        for i in range(emb_b.size(0)):
            pos_i = (allowed_mask[i] > 0.5) & ready
            if pos_i.sum() == 0:
                continue
            if ready.sum() < 2:
                continue
            if pos_i.sum() == ready.sum():
                continue

            sims = torch.matmul(proto_birads[ready], emb_b[i]) / self.tau
            pos_ready_mask = pos_i[ready]

            log_den = torch.logsumexp(sims, dim=0)
            log_num = torch.logsumexp(sims[pos_ready_mask], dim=0)
            loss_i = -(log_num - log_den)

            losses.append(loss_i)
            weights.append(sample_weights[i])

        if len(losses) == 0:
            return torch.tensor(0.0, device=emb_b.device, requires_grad=False)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=emb_b.device, dtype=emb_b.dtype)
        return (losses * weights).sum() / (weights.sum() + 1e-8)


class PrototypeSeparationLoss(nn.Module):
    def __init__(self, margin=0.35):
        super().__init__()
        self.margin = margin

    def forward(self, prototypes, proto_updates):
        ready = (proto_updates >= PROTO_MIN_UPDATES)
        p = prototypes[ready]
        if p.size(0) < 2:
            return torch.tensor(0.0, device=prototypes.device, requires_grad=False)

        sim = torch.matmul(p, p.t())
        eye = torch.eye(sim.size(0), device=sim.device, dtype=torch.bool)
        off = sim[~eye]
        return F.relu(off - self.margin).mean()


# =============================================================================
# Validation / Test
# =============================================================================

@torch.no_grad()
def validate(model, loader, device, cls_loss_fn, class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    total_loss = 0.0
    preds, targets = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        ord_logits = model.forward_encoder(imgs, return_embeddings=False)
        total_loss += cls_loss_fn(ord_logits, labels).item()

        pred = ordinal_logits_to_label(ord_logits)
        preds.extend(pred.cpu().numpy())
        targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    macro_f1 = f1_score(targets, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets, preds, average="weighted", zero_division=0)

    print("\n--- Validation ---")
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))

    return {
        "loss": total_loss / max(len(loader), 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    preds, labels = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        gt = batch["qry_gt"].to(device, non_blocking=True).long()

        ord_logits = model.forward_encoder(imgs, return_embeddings=False)
        pred = ordinal_logits_to_label(ord_logits)

        preds.extend(pred.cpu().numpy())
        labels.extend(gt.cpu().numpy())

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, preds, average="weighted", zero_division=0)
    cm = confusion_matrix(labels, preds)

    print(f"\n{'='*70}")
    print(f"{name} | Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={weighted_f1:.4f}")
    print(cm)
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print('=' * 70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={weighted_f1:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(labels, preds, target_names=class_names, zero_division=0))

    return {"acc": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1}


# =============================================================================
# Training Loop
# =============================================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=5e-5,
    lr_heads=5e-5,
    epochs=60,
    patience=15,
    save_path="best_model.pth",
    w_finding=0.5,
    w_birads=0.5,
    w_hier=0.25,
    w_sep_f=0.02,
    w_sep_b=0.02,
    warmup_epochs=3,
    ramp_epochs=10,
    class_names=None,
):
    class_names = class_names or BIRADS_CLASS_NAMES
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    cls_loss_fn = WeightedCORALLoss(
        num_classes=NUM_CLASSES,
        class_weights=[0.1, 0.15, 1.0, 1.0, 1.2],
        underestimation_weight=2.0,
        level_weights=[1.0, 1.2, 1.5, 1.8],
    ).to(device)

    proto_loss_fn = MultiPositiveProtoInfoNCELoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    hier_loss_fn = HierarchicalBiradsConsistencyLoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    proto_sep_fn = PrototypeSeparationLoss(margin=0.35).to(device)

    head_params = (
        list(model.encoder.cls_head.parameters()) +
        list(model.encoder.proj_head_finding.parameters()) +
        list(model.encoder.proj_head_birads.parameters()) +
        list(model.encoder.p4_proj.parameters()) +
        list(model.encoder.p5_proj.parameters()) +
        list(model.encoder.p5_swin.parameters()) +
        list(model.encoder.p4_from_p5.parameters()) +
        list(model.encoder.p5_from_p4.parameters()) +
        list(model.encoder.global_context.parameters()) +
        list(model.encoder.fuse.parameters()) +
        list(model.encoder.pool.parameters()) +
        list(model.encoder.final_proj.parameters())
    )

    head_param_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.encoder.feature_extractor.parameters() if id(p) not in head_param_ids]

    optimizer = AdamW([
        {
            "params": backbone_params,
            "lr": lr_backbone,
            "weight_decay": 0.05,
        },
        {
            "params": head_params,
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
    ])

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_macro_f1 = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            pw_f = 0.0
            pw_b = 0.0
            pw_h = 0.0
            pw_sf = 0.0
            pw_sb = 0.0
        else:
            ramp = min(1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))
            pw_f = w_finding * ramp
            pw_b = w_birads * ramp
            pw_h = w_hier * ramp
            pw_sf = w_sep_f * ramp
            pw_sb = w_sep_b * ramp

        model.train()
        e_cls = e_pf = e_pb = e_h = e_sf = e_sb = 0.0
        all_preds, all_labels = [], []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            imgs = batch["qry_im"].to(device, non_blocking=True)
            labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                ord_logits, emb_f, emb_b = model.forward_encoder(imgs, return_embeddings=True)

                cls_loss = cls_loss_fn(ord_logits, labels)

                if pw_f > 0:
                    finding_pos_mask = (finding_vec > 0.5).float()

                    finding_sample_w = []
                    for i in range(imgs.size(0)):
                        active = torch.where(finding_vec[i] > 0.5)[0]
                        if len(active) == 0:
                            finding_sample_w.append(0.0)
                        else:
                            sw = sum(FINDING_SAMPLE_WEIGHT[a.item()] for a in active) / len(active)
                            finding_sample_w.append(sw)

                    proto_loss_f = proto_loss_fn(
                        q=emb_f,
                        prototypes=model.proto_finding,
                        proto_updates=model.proto_finding_updates,
                        pos_mask=finding_pos_mask,
                        sample_weights=finding_sample_w,
                    )
                else:
                    proto_loss_f = torch.tensor(0.0, device=device)

                if pw_b > 0:
                    birads_pos_mask = F.one_hot(labels, num_classes=NUM_CLASSES).float()
                    birads_sample_w = [BIRADS_SAMPLE_WEIGHT[lb.item()] for lb in labels]

                    proto_loss_b = proto_loss_fn(
                        q=emb_b,
                        prototypes=model.proto_birads,
                        proto_updates=model.proto_birads_updates,
                        pos_mask=birads_pos_mask,
                        sample_weights=birads_sample_w,
                    )
                else:
                    proto_loss_b = torch.tensor(0.0, device=device)

                if pw_h > 0:
                    allowed_birads_mask = model.build_allowed_birads_mask(finding_vec)
                    hier_sample_w = []
                    for i in range(imgs.size(0)):
                        active = torch.where(finding_vec[i] > 0.5)[0]
                        if len(active) == 0:
                            hier_sample_w.append(0.0)
                        else:
                            sw = sum(FINDING_SAMPLE_WEIGHT[a.item()] for a in active) / len(active)
                            hier_sample_w.append(sw)

                    hier_loss = hier_loss_fn(
                        emb_b=emb_b,
                        proto_birads=model.proto_birads,
                        proto_updates=model.proto_birads_updates,
                        allowed_mask=allowed_birads_mask,
                        sample_weights=hier_sample_w,
                    )
                else:
                    hier_loss = torch.tensor(0.0, device=device)

                if pw_sf > 0:
                    sep_f = proto_sep_fn(model.proto_finding, model.proto_finding_updates)
                else:
                    sep_f = torch.tensor(0.0, device=device)

                if pw_sb > 0:
                    sep_b = proto_sep_fn(model.proto_birads, model.proto_birads_updates)
                else:
                    sep_b = torch.tensor(0.0, device=device)

                total_loss = (
                    cls_loss
                    + pw_f * proto_loss_f
                    + pw_b * proto_loss_b
                    + pw_h * hier_loss
                    + pw_sf * sep_f
                    + pw_sb * sep_b
                )

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                _, emb_f_fresh, emb_b_fresh = model.forward_encoder(imgs, return_embeddings=True)
                model.update_finding_prototypes(emb_f_fresh, finding_vec)
                model.update_birads_prototypes(emb_b_fresh, labels)

            pred = ordinal_logits_to_label(ord_logits)
            all_preds.extend(pred.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

            e_cls += cls_loss.item()
            e_pf += proto_loss_f.item()
            e_pb += proto_loss_b.item()
            e_h += hier_loss.item()
            e_sf += sep_f.item()
            e_sb += sep_b.item()

            pbar.set_postfix(
                cls=f"{cls_loss.item():.3f}",
                pf=f"{proto_loss_f.item():.3f}",
                pb=f"{proto_loss_b.item():.3f}",
                h=f"{hier_loss.item():.3f}",
            )

        n = max(len(train_loader), 1)
        train_acc = accuracy_score(all_labels, all_preds)
        train_macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        train_weighted_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

        print(f"\n{'='*70}")
        print(
            f"Epoch {epoch+1}/{epochs}  "
            f"cls={e_cls/n:.4f}  proto_f={e_pf/n:.4f}  proto_b={e_pb/n:.4f}  "
            f"hier={e_h/n:.4f}  sep_f={e_sf/n:.4f}  sep_b={e_sb/n:.4f}  "
            f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}"
        )

        print(
            f"Proto finding updates: {model.proto_finding_updates.cpu().tolist()}  "
            f"| Proto birads updates: {model.proto_birads_updates.cpu().tolist()}"
        )

        print("\n--- Train ---")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

        val_metrics = validate(
            model=model,
            loader=val_loader,
            device=device,
            cls_loss_fn=cls_loss_fn,
            class_names=class_names,
        )

        print(
            f"Val loss={val_metrics['loss']:.4f}  "
            f"Val acc={val_metrics['acc']:.4f}  "
            f"Val macro_f1={val_metrics['macro_f1']:.4f}  "
            f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
        )
        print('=' * 70)

        scheduler.step()

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch+1}  "
                f"cls={e_cls/n:.4f}  proto_f={e_pf/n:.4f}  proto_b={e_pb/n:.4f}  "
                f"hier={e_h/n:.4f}  sep_f={e_sf/n:.4f}  sep_b={e_sb/n:.4f}  "
                f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}  "
                f"train_weighted_f1={train_weighted_f1:.4f}  "
                f"val_loss={val_metrics['loss']:.4f}  val_acc={val_metrics['acc']:.4f}  "
                f"val_macro_f1={val_metrics['macro_f1']:.4f}  "
                f"val_weighted_f1={val_metrics['weighted_f1']:.4f}\n"
            )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_macro_f1": best_macro_f1,
            }, save_path)
            print(f"Saved best model macro-F1={best_macro_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_macro_f1


# =============================================================================
# Runner
# =============================================================================

def run_experiments(
    train_loader,
    val_loader,
    test_loader,
    inbreast_loader,
):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_config = dict(
        lr_backbone   = 5e-5,
        lr_heads      = 5e-5,
        epochs        = 60,
        patience      = 15,
        w_finding     = 0.5,
        w_birads      = 0.5,
        w_hier        = 0.25,
        w_sep_f       = 0.02,
        w_sep_b       = 0.02,
        warmup_epochs = 3,
        ramp_epochs   = 10,
        class_names   = BIRADS_CLASS_NAMES,
    )

    backbones = [
        {
            "name": "efficientnet_b3",
            "fn":   models.efficientnet_b3,
            "w":    models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn":   models.convnext_base,
            "w":    models.ConvNeXt_Base_Weights.DEFAULT,
        },
    ]

    all_results = {}

    for cfg in backbones:
        out_dir = f"/workspace/Thesis_updated_results/birads_512_hybrid_p4p5_crossscale_global/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(f"Backbone: {cfg['name']}")
        print(f"{'#'*70}")

        model = DualProtoHybridNet(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.4,
            num_classes=NUM_CLASSES,
            num_findings=NUM_FINDINGS,
            temperature=0.07,
            fuse_dim=256,
            swin_heads=4,
            swin_window=7,
            swin_depth=2,
            cross_heads=4,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable params: {num_params:,}")
        print(f"Finding proto momentums: {FINDING_PROTO_MOMENTUM}")
        print(f"BI-RADS proto momentums: {BIRADS_PROTO_MOMENTUM}")

        best_macro_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **train_config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

        vindr_metrics = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=BIRADS_CLASS_NAMES,
        )

        inbreast_metrics = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=BIRADS_CLASS_NAMES,
        )

        all_results[cfg["name"]] = dict(
            best_val_macro_f1=best_macro_f1,
            vindr_acc=vindr_metrics["acc"],
            vindr_macro_f1=vindr_metrics["macro_f1"],
            vindr_weighted_f1=vindr_metrics["weighted_f1"],
            inbreast_acc=inbreast_metrics["acc"],
            inbreast_macro_f1=inbreast_metrics["macro_f1"],
            inbreast_weighted_f1=inbreast_metrics["weighted_f1"],
        )

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*85}")
    print("FINAL RESULTS")
    print(f"{'='*85}")
    print(
        f"{'Model':<22} "
        f"{'Val Macro-F1':>12} "
        f"{'VinDr Macro-F1':>15} "
        f"{'INbreast Macro-F1':>18}"
    )
    print("-" * 85)
    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_macro_f1']:>12.4f} "
            f"{r['vindr_macro_f1']:>15.4f} "
            f"{r['inbreast_macro_f1']:>18.4f}"
        )

    return all_results


results = run_experiments(
    train_loader=tr_dl,
    val_loader=val_dl,
    test_loader=ts_dl,
    inbreast_loader=inbreast_dl,
)

Device: cuda

######################################################################
Backbone: efficientnet_b3
######################################################################
Trainable params: 16,241,709
Finding proto momentums: [0.999, 0.997, 0.98, 0.96, 0.92, 0.85]
BI-RADS proto momentums: [0.999, 0.997, 0.97, 0.96, 0.92]


Epoch 1/60: 100%|██████████| 441/441 [06:52<00:00,  1.07it/s, cls=0.466, h=0.000, pb=0.000, pf=0.000] 


Epoch 1/60  cls=0.8109  proto_f=0.0000  proto_b=0.0000  hier=0.0000  sep_f=0.0000  sep_b=0.0000  train_acc=0.1436  train_macro_f1=0.1866
Proto finding updates: [441, 436, 422, 318, 183, 71]  | Proto birads updates: [441, 436, 356, 364, 282]

--- Train ---
              precision    recall  f1-score   support

   BI-RADS 1       0.33      0.00      0.00      3203
   BI-RADS 2       0.30      0.04      0.06      1957
   BI-RADS 3       0.11      0.65      0.18       704
   BI-RADS 4       0.13      0.40      0.20       758
   BI-RADS 5       0.56      0.42      0.48       434

    accuracy                           0.14      7056
   macro avg       0.29      0.30      0.19      7056
weighted avg       0.30      0.14      0.09      7056




--- Validation ---
              precision    recall  f1-score   support

   BI-RADS 1       0.00      0.00      0.00       538
   BI-RADS 2       0.00      0.00      0.00       196
   BI-RADS 3       0.03      0.87      0.05        23
   BI-RADS 4       0.10      0.17      0.12        23
   BI-RADS 5       0.50      0.57      0.53         7

    accuracy                           0.04       787
   macro avg       0.12      0.32      0.14       787
weighted avg       0.01      0.04      0.01       787

Val loss=0.7788  Val acc=0.0356  Val macro_f1=0.1418  Val weighted_f1=0.0099
Saved best model macro-F1=0.1418


Epoch 2/60: 100%|██████████| 441/441 [05:13<00:00,  1.41it/s, cls=0.677, h=0.000, pb=0.000, pf=0.000]


...

Epoch 39/60: 100%|██████████| 441/441 [05:12<00:00,  1.41it/s, cls=0.118, h=0.281, pb=0.283, pf=0.559]



Epoch 39/60  cls=0.1043  proto_f=0.6253  proto_b=0.2145  hier=0.1719  sep_f=0.2274  sep_b=0.0976  train_acc=0.8566  train_macro_f1=0.9012
Proto finding updates: [17199, 17098, 16676, 12274, 7153, 2522]  | Proto birads updates: [17199, 17101, 14049, 14297, 10859]

--- Train ---
              precision    recall  f1-score   support

   BI-RADS 1       0.87      0.84      0.85      3224
   BI-RADS 2       0.75      0.77      0.76      1970
   BI-RADS 3       0.96      0.97      0.97       714
   BI-RADS 4       0.95      0.96      0.95       716
   BI-RADS 5       0.96      0.98      0.97       432

    accuracy                           0.86      7056
   macro avg       0.90      0.90      0.90      7056
weighted avg       0.86      0.86      0.86      7056


--- Validation ---
              precision    recall  f1-score   support

   BI-RADS 1       0.85      0.86      0.85       538
   BI-RADS 2       0.53      0.58      0.55       196
   BI-RADS 3       0.29      0.09      0.13      

Epoch 40/60: 100%|██████████| 441/441 [05:59<00:00,  1.23it/s, cls=0.039, h=0.118, pb=0.118, pf=0.776]



Epoch 40/60  cls=0.0886  proto_f=0.6729  proto_b=0.1877  hier=0.1562  sep_f=0.2593  sep_b=0.0937  train_acc=0.8586  train_macro_f1=0.9048
Proto finding updates: [17640, 17537, 17100, 12580, 7336, 2589]  | Proto birads updates: [17640, 17540, 14402, 14651, 11134]

--- Train ---
              precision    recall  f1-score   support

   BI-RADS 1       0.88      0.84      0.86      3316
   BI-RADS 2       0.74      0.79      0.76      1932
   BI-RADS 3       0.96      0.98      0.97       697
   BI-RADS 4       0.96      0.96      0.96       684
   BI-RADS 5       0.97      0.98      0.98       427

    accuracy                           0.86      7056
   macro avg       0.90      0.91      0.90      7056
weighted avg       0.86      0.86      0.86      7056


--- Validation ---
              precision    recall  f1-score   support

   BI-RADS 1       0.83      0.91      0.87       538
   BI-RADS 2       0.59      0.49      0.54       196
   BI-RADS 3       0.14      0.04      0.07      

Epoch 41/60:   7%|▋         | 29/441 [00:28<05:39,  1.21it/s, cls=0.129, h=0.102, pb=0.287, pf=0.638]